# Abstract
Since I wrote [my book on Kaggle × LLM](https://www.amazon.co.jp/Kaggle%E3%81%A7%E3%81%AF%E3%81%98%E3%82%81%E3%82%8B%E5%A4%A7%E8%A6%8F%E6%A8%A1%E8%A8%80%E8%AA%9E%E3%83%A2%E3%83%87%E3%83%AB%E5%85%A5%E9%96%80-%E8%87%AA%E7%84%B6%E8%A8%80%E8%AA%9E%E5%87%A6%E7%90%86%E3%80%88%E5%AE%9F%E8%B7%B5%E3%80%89%E3%83%97%E3%83%AD%E3%82%B0%E3%83%A9%E3%83%9F%E3%83%B3%E3%82%B0-KS%E6%83%85%E5%A0%B1%E7%A7%91%E5%AD%A6%E5%B0%82%E9%96%80%E6%9B%B8-%E9%AB%98%E9%87%8E-%E6%B5%B7%E6%96%97/dp/4065415241), there haven’t been any competitions for simple text classification tasks, so I’ve had limited opportunities to put the book to the test. That’s why I’m really excited that a competition like this is being held.
I’ve adapted the notebooks featured in the book for this competition, so I hope you’ll find them useful.
This notebook performs training and inference using Gemma2B.


In [1]:
!pip install --no-index --find-links=/kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels \
    "transformers==4.51.3" "bitsandbytes==0.45.5" "accelerate==1.6.0" "peft==0.15.2"

Looking in links: /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels
Processing /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels/transformers-4.51.3-py3-none-any.whl
Processing /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels/bitsandbytes-0.45.5-py3-none-manylinux_2_24_x86_64.whl
Processing /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels/accelerate-1.6.0-py3-none-any.whl
Processing /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels/peft-0.15.2-py3-none-any.whl
Processing /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/wheels/tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from transformers==4.51.3)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4

# Import Libraries

In [2]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

E0000 00:00:1792034923.832510      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1792034923.870955      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1792034924.191975      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792034924.192001      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792034924.192004      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792034924.192006      22 computation_placer.cc:177] computation placer already registered. Please check linka

# Config

In [3]:
class CFG:
    VER = 1
    AUTHOR = "takaito"
    COMPETITION = "NameThatLLM"
    DATA_PATH = Path("/kaggle/input/competitions/name-that-llm")
    MODEL_PATH = "/kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/gemma-2-2b-it-bnb-4bit"
    MODEL_NAME = "gemma-2-2b-it-bnb-4bit"
    MAX_LENGTH = 2048
    EPOCHS = 3
    STEPS = 40
    WARMUP_RATIO = 0.1
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-4
    OPTIM = "adamw_8bit"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 3
    TARGET_COL = "model"
    TARGET_COL_CLASS_NUM = 18
    METRIC = "acc"
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"

In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

In [5]:
TARGET_MODELS = [
    # OpenAI
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    # Google
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    # Anthropic
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    # Qwen
    "qwen3.8-max",
    # ZAI
    "glm-5.3",
    "glm-5.3-flash",
    # Moonshot AI
    "kimi-k3",
]

model2id = {m: i for i, m in enumerate(TARGET_MODELS)}
id2model = {i: m for m, i in model2id.items()}

# Seed Setting

In [6]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if CFG.USE_GPU:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)

In [7]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.use_deterministic_algorithms(True)

# Read Data

In [8]:
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

In [9]:
train_df["labels"] = train_df[CFG.TARGET_COL].map(model2id)

# Tokenizer Setting

In [10]:
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH)
tokenizer.add_eos_token = True
tokenizer.padding_side = "right"

def tokenize(example, tokenizer):
    return tokenizer(example["text"], add_special_tokens=True)

# Make Dataset

In [11]:
def mk_dataset(df, tokenizer):
    raw_datasets = Dataset.from_pandas(df)
    tokenized_datasets = raw_datasets.map(
        tokenize, 
        fn_kwargs={"tokenizer": tokenizer}
    ).remove_columns(["text"])
    return tokenized_datasets

In [12]:
ds = mk_dataset(train_df[["text", "labels"]], tokenizer)
ds_test = mk_dataset(test_df[["text"]], tokenizer)

Map:   0%|          | 0/2699 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

# Training & Inference

In [13]:
def compute_metrics(p):
    label_preds, labels = p
    label_prob_preds = torch.softmax(torch.tensor(label_preds), dim = 1).numpy()
    score = accuracy_score(labels, label_prob_preds.argmax(axis=1))
    return {"acc": score}

In [14]:
def get_args(CFG, fold):
    args = TrainingArguments(
        output_dir=f"{CFG.PREFIX}-fold{fold}",
        report_to="none",
        eval_strategy="steps",
        logging_strategy="steps",
        save_strategy="steps",
        eval_steps=CFG.STEPS,
        logging_steps=CFG.STEPS,
        save_steps=CFG.STEPS,
        save_total_limit=1,
        metric_for_best_model=CFG.METRIC,
        greater_is_better=CFG.METRIC_MAXIMIZE_FLAG,
        optim=CFG.OPTIM,
        learning_rate=CFG.LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_ratio=CFG.WARMUP_RATIO,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.TRAIN_BATCH_SPLIT,
        num_train_epochs=CFG.EPOCHS,
        fp16=True,
        seed=CFG.SEED,
        data_seed=CFG.SEED,
    )
    return args

In [15]:
def setting_model(CFG):
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.MODEL_PATH,
        num_labels=CFG.TARGET_COL_CLASS_NUM,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    return model

In [16]:
predictions = np.zeros((len(train_df), CFG.TARGET_COL_CLASS_NUM))
test_predictions = np.zeros((len(test_df), CFG.TARGET_COL_CLASS_NUM))
kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=CFG.SEED)
for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df[CFG.TARGET_COL])):
    model = setting_model(CFG)
    trainer = Trainer(
        args=get_args(CFG, fold),
        model=model,
        train_dataset=ds.select(train_index),
        eval_dataset=ds.select(valid_index),
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    # trainer.save_model(f"{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")
    # tokenizer.save_pretrained(f"{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")
    valid_preds = trainer.predict(ds.select(valid_index)).predictions
    predictions[valid_index] = torch.softmax(torch.tensor(valid_preds), dim=1).numpy()
    test_preds = trainer.predict(ds_test).predictions
    test_predictions += torch.softmax(torch.tensor(test_preds), dim=1).numpy() / CFG.N_SPLIT
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/gemma-2-2b-it-bnb-4bit and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss,Acc
40,3.274700,2.298912,0.204444
80,1.769400,1.971023,0.273333


Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/gemma-2-2b-it-bnb-4bit and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss,Acc
40,3.597400,2.527158,0.141111
80,1.929600,1.994032,0.230000


Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/gemma-2-2b-it-bnb-4bit and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss,Acc
40,3.582800,2.309578,0.187987
80,1.856900,2.001108,0.258065


In [17]:
print("CV Score: ", accuracy_score(train_df["labels"], predictions.argmax(axis=1)))

CV Score:  0.25120414968506855


In [18]:
test_df["labels"] = test_predictions.argmax(axis=1)
test_df["model"] = test_df["labels"].map(id2model)
test_df[["id", "model"]].to_parquet('submission.parquet', index=False)